# Display spikes that were automatically detected

## Import recordings

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy
from scipy import signal
import os
from ephyviewer import mkQApp, MainViewer, TraceViewer
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, InMemoryAnalogSignalSource, EpochEncoder,TimeFreqViewer,AnalogSignalSourceWithScatter, EpochEncoder_ABC
from matplotlib import cm
from matplotlib.colors import to_hex
import re
import mne
import pandas as pd
import numpy as np
import csv
from collections import defaultdict
import ast
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import string
import pickle

%matplotlib widget

Choose .edf file

In [ ]:
file= 'AGAL06-E_06032017' #AGAL06-E_06032017 #GUNO03-E_24042018 #AZSA10-E_12112018

Load cleaned edf file (no montage)

In [ ]:
dpath = f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}.edf"
folder_base = Path(dpath) 
data = mne.io.read_raw_edf(f'{Path(dpath).parent}/{Path(dpath).name[:-4]}_repaired_montage.edf', preload=False)
raw_data = data.get_data() #data = mne.io.read_raw_edf(dpath, encoding='latin1')

info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

Zscore & filter signal

In [ ]:
montage_eeg=[]
montage_name=[]
for i, ch in enumerate(channels):
    if ch.count('-')==0:
        ch1 = ch
        eeg=combined[:,i]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
    else:        
        ch1 = ch
        eeg=combined[:,i]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 50.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
montage_name_eeg=montage_name[:-2]

## Display all spikes detected by SpikeDet (montage) 

Load .mat file containing detected spike

In [ ]:
mat = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/spikes_spmeeg_{file}_repaired_montage.mat", squeeze_me=True, struct_as_record=False)
data = mat['D'].trials.events
result = {}
for s in data:
    times = np.asarray(s.time).flatten()
    ch_name = str(np.asarray(s.value).flatten()[0]).replace("EEG_", "")
    if ch_name not in result:
        result[ch_name] = []
    result[ch_name].extend(times.tolist())
ordered_result = {k: result[k] for k in montage_name if k in result}

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(ordered_result.values())
}

order_index = {k: i for i, k in enumerate(montage_name)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(ordered_result.keys())
    if k in order_index
}

With sleep scoring + spikes (montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels, scatter_colors= {i: "#FF000084" for i in range(len(montage_name))},channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by SpikeDet (montage) + Global clustering  

In [ ]:
matsp = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/sp_spikes_spmeeg_{file}_repaired_montage.mat", squeeze_me=True, struct_as_record=False)
GlobalSpike = matsp['GlobalSpikes']

clusters = np.array(GlobalSpike.Clusters.ClusterIndices)    
nb_cluster = int(clusters.max())
nb_cluster_tot = nb_cluster * len(montage_name_eeg)
print(f"{nb_cluster} clusters found")

scatter_channels = defaultdict(list)
scatter_indexes = defaultdict(list)

for clus in np.arange(0, nb_cluster_tot, 1):
    scatter_channels[clus+1]=[np.repeat(np.arange(0,len(montage_name_eeg)), nb_cluster)[clus]]
    scatter_indexes[clus+1]=[]

for spike in np.arange(0, len(clusters), 1):    
    key = int(clusters[spike] + (GlobalSpike.GenDet.SpikesInChannel[spike] * nb_cluster)- nb_cluster)
    scatter_indexes[key].append(GlobalSpike.GenDet.Epoch[spike])
    
scatter_channels = dict(sorted(scatter_channels.items()))
scatter_indexes = {k: pd.Series(v) for k, v in scatter_indexes.items()}
scatter_indexes = dict(sorted(scatter_indexes.items()))

groups = {}
for k, v in scatter_channels.items():
    groups.setdefault(v[0], []).append(k)
color_dict = {}
cmap = matplotlib.colormaps['rainbow']
for group_keys in groups.values():
    n = len(group_keys)
    norm = mcolors.Normalize(vmin=0, vmax=n-1)
    for i, key in enumerate(group_keys):
        color_dict[key] = mcolors.to_hex(cmap(norm(i))) + "80"

scatter_indexesO = scatter_indexes.copy()
scatter_channelsO = scatter_channels.copy()
color_dictO = color_dict.copy()


SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
SleepScoring = pd.read_csv(SleepScoring_filename)

Remove spikes detected during Wake or Unstaged

In [ ]:
def find_stage(time_value, df):
    df['end_time'] = df['time'] + df['duration']
    row = df[(df['time'] <= time_value) & (time_value < df['end_time'])]
    if len(row) == 0:
        return None
    return row.iloc[0]['label']

stage_dict = {}
for key, time_list in scatter_indexesO.items():
    stage_dict[key] = [find_stage(t, SleepScoring) for t in time_list/samplerate]
    
labels_to_remove = {None, 'WAKE'}
scatter_indexes = {}
for key in scatter_indexesO:
    times  = scatter_indexesO[key]
    stages = stage_dict[key]
    filtered_times = [
        t for t, s in zip(times, stages)
        if s not in labels_to_remove
    ]    
    scatter_indexes[key] = filtered_times
scatter_indexes = {k: pd.Series(v) for k, v in scatter_indexes.items()}
scatter_indexes = dict(sorted(scatter_indexes.items()))
scatter_indexesO=scatter_indexes.copy()

Select a cluster (optional)

In [ ]:
sel_cluster = 2  # 1 = purple, 2 = blue, 3 =  green, 4 = orange, 5 = red
sel_clusters= np.arange(sel_cluster, nb_cluster_tot, nb_cluster)
scatter_indexes = {k: v for k, v in scatter_indexesO.items() if k in sel_clusters}
scatter_channels= {k: v for k, v in scatter_channelsO.items() if k in sel_clusters}
color_dict= {k: v for k, v in color_dictO.items() if k in sel_clusters}
print(f"Cluster {sel_cluster} contains {sum(len(v) for v in scatter_indexes.values())} detected spikes")

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels,  scatter_colors = color_dict, channel_names=montage_name)
view1 = TraceViewer(source=source)
#scatter_colors = color_dict,
view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by SpikeDet (montage) + Clustering per channel

Load .mat file containing cluster identity of detected spike

In [ ]:
matsp = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/sp_spikes_spmeeg_{file}_repaired_montage.mat", squeeze_me=True, struct_as_record=False)

sp_array = matsp['sp']
montage_name_eeg=montage_name[:-2]
SpikeClusters = {}
rand_val = 0
scatter_channels = {}
epoch_dict={}
for i, channel in enumerate(montage_name_eeg):
    clusters = np.array(sp_array[i].Clusters.ClusterIndices)    
    max_cluster = int(clusters.max())
    SpikeClusters[channel] = (clusters + rand_val).tolist()
    epoch_dict[channel] = np.array(sp_array[i].GenDet.Epoch)
    for j in range(rand_val, rand_val + max_cluster + 1):
        scatter_channels[j] = [i]
    rand_val += max_cluster + 1
    
scatter_indexes = {}
temp_dict = defaultdict(list)
for key in SpikeClusters:
    vals_A = SpikeClusters[key]
    vals_B = np.asarray(epoch_dict[key]).ravel()
    vals_B = np.floor(vals_B).astype(int)
    for a_val, b_val in zip(vals_A, vals_B):
        temp_dict[a_val].append(b_val)
scatter_indexes = dict(temp_dict)
scatter_indexes = {k: pd.Series(v) for k, v in scatter_indexes.items()}
scatter_indexes = dict(sorted(scatter_indexes.items()))

groups = {}
for k, v in scatter_channels.items():
    groups.setdefault(v[0], []).append(k)
color_dict = {}
cmap = matplotlib.colormaps['rainbow']
for group_keys in groups.values():
    n = len(group_keys)
    norm = mcolors.Normalize(vmin=0, vmax=n-1)
    for i, key in enumerate(group_keys):
        color_dict[key] = mcolors.to_hex(cmap(norm(i))) + "80"

Select a cluster in a channel (optional)

In [ ]:
sel_cluster = 8
scatter_indexes= {k: v for k, v in scatter_indexes.items() if k == sel_cluster}
scatter_channels= {k: v for k, v in scatter_channels.items() if k == sel_cluster}
color_dict= {k: v for k, v in color_dict.items() if k == sel_cluster}

With sleep scoring + spikes & cluster ID (montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels,  scatter_colors = color_dict, channel_names=montage_name)
view1 = TraceViewer(source=source)
#scatter_colors = color_dict,
view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by Snooz (montage)

In [ ]:
events_dict = defaultdict(list)
with open(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/{file}_repaired_montage.tsv", newline='') as f:
    reader = csv.DictReader(f, delimiter="\t", quotechar='"')
    for row in reader:
        start = float(row['start_sec'])
        channels = ast.literal_eval(row['channels'])  # Convert string list to Python list
        for ch in channels:
            events_dict[ch].append(start)

# Convert defaultdict to normal dict
events_dict = dict(events_dict)
print(events_dict)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(events_dict.values())
}

order_index = {k: i for i, k in enumerate(montage_name)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(events_dict.keys())
    if k in order_index
}

SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels, scatter_colors= {i: "#FF000084" for i in range(len(montage_name))},channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'


## Display spikes detected by SpikeDect (single channel) 

Load .mat file

In [ ]:
mat = scipy.io.loadmat(f"C:/Users/Manip2/Documents/EpiKid/ScoredSTS/spikes_spmeeg_{file}_repaired.mat", squeeze_me=True, struct_as_record=False)
data = mat['D'].trials.events
result = {}
for s in data:
    times = np.asarray(s.time).flatten()
    value_name = str(np.asarray(s.value).flatten()[0]).replace("EEG_", "")

    if value_name not in result:
        result[value_name] = []
    result[value_name].extend(times.tolist())
ordered_result = {k: result[k] for k in channels if k in result}

With sleep scoring + spikes (no montage)

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

scatter_indexes = {
    i: pd.Series(np.asarray(v).ravel() * samplerate, name="start time", dtype=int)
    for i, v in enumerate(ordered_result.values())
}

order_index = {k: i for i, k in enumerate(channels)}

scatter_channels = {
    idx: [order_index[k]]
    for idx, k in enumerate(ordered_result.keys())
    if k in order_index
}

SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(eeg, samplerate, t_start, scatter_indexes, scatter_channels, channel_names=channels)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(channels): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'
